In [12]:
# 不同深度的扩散因子

import pandas as pd 
import os
import ast
import numpy as np


def  process_effective_factor(file1):
        
  
    df1 = pd.read_csv(file1)
    df1 = df1.groupby('source_file', sort=False).agg(list).reset_index()

    # depth 0~5, 最后一层肯定是没有，因为，最后一层没有下一层了
    a, b, c, d, e, f = [],[],[],[],[],[]
    for i in range(len(df1)):
        average_b = df1.iloc[i,6]
        # print(average_b)

        for j in range(len(average_b)-1):

            if j == 0:
                a.append(average_b[j])
            if j == 1:
                b.append(average_b[j])
            if j == 2:
                c.append(average_b[j])
            if j == 3:
                d.append(average_b[j])
            if j == 4:
                e.append(average_b[j])
            if j == 5:
                f.append(average_b[j])

        # print('*'*10)
        # print(ratio)
        # break

        
    return a, b, c, d, e, f


def empety_list(lst):
    if len(lst) == 0:
        return [0]
    else:
        return lst


data_types = ['50k_class_0', '50k_class_1', '50k_class_3',  '50k_class_5', '50k_class_10', \
              '400k_class_0', '400k_class_1', '400k_class_3',  '400k_class_5', '400k_class_10']
# data_types = [ '400k_class_10']
# data_types = [ '50k_kd_class_10']


file_specifics = ['experiment_results.csv']

for data_type in data_types:
    dir = 'result_retroformer_2026_03_03/'+data_type+'_effectivate_factor_analysis_0.85'
    print('*'*70)
    print(data_type+'_effectivate_factor_analysis_0.85')
    a, b, c, d, e, f = process_effective_factor(dir+'/'+file_specifics[0])

    print(round(np.mean(empety_list(a)), 2), round(np.mean(empety_list(b)), 2), round(np.mean(empety_list(c)), 2),\
           round(np.mean(empety_list(d)), 2), round(np.mean(empety_list(e)), 2), round(np.mean(empety_list(f)), 2))
#     break
print('done\n')    
# print('a: ',a)
# print('b: ',b)
# print('c: ',c)   
# print('d: ',d)
# print('e: ',e)
# print('f: ',f)

for data_type in data_types:
    dir = 'result_retroformer_2026_03_03/'+data_type+'_effectivate_factor_analysis_no_dediversity'
    print('*'*70)
    print(data_type+'_effectivate_factor_analysis_no_dediversity')
    a, b, c, d, e, f =  process_effective_factor(dir+'/'+file_specifics[0])

    print(round(np.mean(empety_list(a)), 2), round(np.mean(empety_list(b)), 2), round(np.mean(empety_list(c)), 2),\
           round(np.mean(empety_list(d)), 2), round(np.mean(empety_list(e)), 2), round(np.mean(empety_list(f)), 2))
    # break
print('done\n')  


**********************************************************************
50k_class_0_effectivate_factor_analysis_0.85
9.68 7.14 1.52 0.07 0.02 0.0
**********************************************************************
50k_class_1_effectivate_factor_analysis_0.85
5.35 4.23 1.23 0.07 0.12 0.0
**********************************************************************
50k_class_3_effectivate_factor_analysis_0.85
8.6 5.98 1.49 0.06 0.14 0.07
**********************************************************************
50k_class_5_effectivate_factor_analysis_0.85
10.04 6.84 1.77 0.08 0.01 0.0
**********************************************************************
50k_class_10_effectivate_factor_analysis_0.85
12.53 8.35 2.45 0.12 0.03 0.13
**********************************************************************
400k_class_0_effectivate_factor_analysis_0.85
3.31 2.18 0.36 0.03 0.0 0.0
**********************************************************************
400k_class_1_effectivate_factor_analysis_0.85
1.93 1.5 

In [ ]:
# 由于是动态过程，各层作为最后一层时，所有节点都是可以构成路径的, 同时容易忽略的一点是传播到对于某一层时，
# 虽然其所有节点都是对应路径的，但是有些路径可能在上一层已经终止了，所以所有路径等于这一层的节点数（这一层所有路径+浅层所有终止的路径）
# 这一层的路径在experiment_results.csv对应的参数是n_nodes
# 浅层次的路径是n_nodes-n_parents (n_parents是指这一层节点后来作为parents进行了扩充)，差值就是这一层的叶节点数
#source_file,depth,n_nodes,n_parents,avg_b_eff,n_reaction_types

# 累积有效因子[10.0, 75.294, 535.7469275999999, 0.0]  # 总长度为3，最后终止了，所以有效分子因子是0，beff = N(depth+1)/N(depth) = 0
# 每层的节点数[1, 22, 1074, 6953], 使用[22, 1074, 6953]+浅层的叶节点数（终止节点）

In [ ]:
import pandas as pd 
import os
import ast
import numpy as np


def process_all(file1):
        
  
    df1 = pd.read_csv(file1)
    depth = df1['depth'].max()
    # print('max depth:', depth)

    df1 = df1.groupby('source_file', sort=False).agg(list).reset_index()

    a, b, c, d, e, f = [],[],[],[],[],[]
    for i in range(len(df1)):
        n_nodes = df1.iloc[i,2]
        n_nodes_as_parents = df1.iloc[i,3]
        n_nodes_as_leaf_nodes = df1.iloc[i,4]
        # print('n_nodes: ',n_nodes)
        # print(n_nodes_as_leaf_nodes)
        # 每层的节点数[1, 22, 1074, 6953], 使用[22, 1074, 6953]+浅层的叶节点数（终止节点）
        for j in range(len(n_nodes)-1,0,-1):
            # print(j)
            # print(len(n_nodes))
            shallow_paths = sum(n_nodes_as_leaf_nodes[:j])
            # print('shallow_paths', shallow_paths)
            n_nodes[j] += shallow_paths

        # print('n_nodes: ',n_nodes)
        average_b = df1.iloc[i,6]
        # break

        # 累积扩散因子
        # print('average_b: ',average_b)
        for j in range(1,len(average_b)):
            average_b[j] = average_b[j] * average_b[j-1]
        # print('average_b: ',average_b)
        
        # print('*'*10)
        # print(average_b)
        # break
        
        ratio = []
        for j in range(len(n_nodes)-1):
            if n_nodes[j+1] != 0:
                ratio.append(n_nodes[j+1] / average_b[j])
 
            if j == 0:
                a.append(ratio[j])
            if j == 1:
                b.append(ratio[j])
            if j == 2:
                c.append(ratio[j])
            if j == 3:
                d.append(ratio[j])
            if j == 4:
                e.append(ratio[j])
            if j == 5:
                f.append(ratio[j])

        # print('*'*10)
        # print(ratio)
        # break

        
    return a, b, c, d, e, f


def empety_list(lst):
    if len(lst) == 0:
        return [0]
    else:
        return lst


data_types = ['50k_class_0', '50k_class_1', '50k_class_3',  '50k_class_5', '50k_class_10', \
              '400k_class_0', '400k_class_1', '400k_class_3',  '400k_class_5', '400k_class_10']
# data_types = [ '400k_class_10']
# data_types = [ '50k_kd_class_10']


file_specifics = ['experiment_results.csv']

for data_type in data_types:
    dir = 'result_retroformer_2026_03_03/'+data_type+'_effectivate_factor_analysis_0.85'
    print('*'*70)
    print(data_type+'_effectivate_factor_analysis_0.85')
    a, b, c, d, e, f = process_all(dir+'/'+file_specifics[0])

    print(round(np.mean(empety_list(a)), 2), round(np.mean(empety_list(b)), 2), round(np.mean(empety_list(c)), 2),\
           round(np.mean(empety_list(d)), 2), round(np.mean(empety_list(e)), 2), round(np.mean(empety_list(f)), 2))
    # break
print('done\n')    
# print('a: ',a)
# print('b: ',b)
# print('c: ',c)   
# print('d: ',d)
# print('e: ',e)
# print('f: ',f)

for data_type in data_types:
    dir = 'result_retroformer_2026_03_03/'+data_type+'_effectivate_factor_analysis_no_dediversity'
    print('*'*70)
    print(data_type+'_effectivate_factor_analysis_no_dediversity')
    a, b, c, d, e, f = process_all(dir+'/'+file_specifics[0])

    print(round(np.mean(empety_list(a)), 2), round(np.mean(empety_list(b)), 2), round(np.mean(empety_list(c)), 2),\
           round(np.mean(empety_list(d)), 2), round(np.mean(empety_list(e)), 2), round(np.mean(empety_list(f)), 2))
    # break
print('done\n')  

**********************************************************************
50k_class_0_effectivate_factor_analysis_0.85
5.14 21.62 103.86 32703.53 164214.82 0.0
**********************************************************************
50k_class_1_effectivate_factor_analysis_0.85
8.32 56.63 413.89 106873.74 4676816.51 0.0
**********************************************************************
50k_class_3_effectivate_factor_analysis_0.85
5.18 23.83 121.68 15493.36 186912.94 5583553.22
**********************************************************************
50k_class_5_effectivate_factor_analysis_0.85
4.2 16.16 68.86 4003.44 111973.9 0.0
**********************************************************************
50k_class_10_effectivate_factor_analysis_0.85
2.98 8.53 26.96 1327.63 13541.82 6949.74
**********************************************************************
400k_class_0_effectivate_factor_analysis_0.85
20.52 326.83 6100.11 3244949.66 0.0 0.0
****************************************************